# Figure 3 — Epicenter frequency maps and connectivity metrics

- **SubA**: Cortical maps of epicenter frequency (proportion of ASD participants for whom each region is the minimum-GOF epicenter).
- **SubB**: Comparative frequency maps between Subtype L and Subtype H.
- **SubC**: Violin plots of network connectivity metrics (e.g., node strength, betweenness) across groups.

## Panel SubA — Epicenter frequency cortical map

In [ ]:
"""Panel A: cortical frequency maps of the individual-level epicenters for ABIDE-II and CABIC."""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colorbar as mcolorbar
from enigmatoolbox.utils.parcellation import parcel_to_surface
from enigmatoolbox.plotting import plot_cortical

# Optional headless display (only needed on servers without a display)
import subprocess, time
xvfb_process = None
try:
    display_num = 99
    xvfb_process = subprocess.Popen(
        ['Xvfb', f':{display_num}', '-screen', '0', '1920x1080x24'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(0.5)
    os.environ['DISPLAY'] = f':{display_num}'
    print(f"[Xvfb] started (:{display_num})")
except Exception as e:
    print(f"[Xvfb] could not start: {e}")

# --- Global style configuration ---
GLOBAL_CONFIG = {
    'DPI': 300, 'FIGURE_WIDTH': 16, 'FIGURE_HEIGHT': 6,
    'FONT_SIZE_MAIN': 24, 'FONT_SIZE_LABEL': 28, 'FONT_SIZE_TICK': 22,
}
SAVE_DIR = 'Fig3'
os.makedirs(SAVE_DIR, exist_ok=True)
plt.rcParams.update({'font.size': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'axes.titlesize': GLOBAL_CONFIG['FONT_SIZE_MAIN'],
                     'font.family': 'serif', 'font.serif': ['Times New Roman'],
                     'savefig.dpi': GLOBAL_CONFIG['DPI']})

# --- Tasks: cohort frequency maps (outputs of the Epicenter notebooks) ---
tasks = [
    {"file": "output/Center/ASD_Total_Best_Seeds_MinGOF_aparc.csv",
     "column": "Frequency_Total_ASD", "title": "ABIDE2", "temp_name": "temp_abide2.png"},
    {"file": "output/Center/CABIC_ASD_Total_Best_Seeds_MinGOF_aparc.csv",
     "column": "Frequency_Total_ASD", "title": "CABIC", "temp_name": "temp_cabic.png"},
]

# Custom sequential colormap for frequency
custom_cmap_name = 'frequency_cmap'
colors_list = ['#f7fbff', '#deebf7', '#9ecae1', '#4292c6', '#08306b']
freq_cmap = LinearSegmentedColormap.from_list(custom_cmap_name, colors_list)
try:
    plt.colormaps.register(cmap=freq_cmap, name=custom_cmap_name, force=True)
except Exception:
    pass

# Desikan-Killiany 68 region labels in enigmatoolbox fsa5 order
dk68_labels = [
    'lh_bankssts', 'lh_caudalanteriorcingulate', 'lh_caudalmiddlefrontal', 'lh_cuneus',
    'lh_entorhinal', 'lh_frontalpole', 'lh_fusiform', 'lh_inferiorparietal',
    'lh_inferiortemporal', 'lh_insula', 'lh_isthmuscingulate', 'lh_lateraloccipital',
    'lh_lateralorbitofrontal', 'lh_lingual', 'lh_medialorbitofrontal', 'lh_middletemporal',
    'lh_parahippocampal', 'lh_paracentral', 'lh_parsopercularis', 'lh_parsorbitalis',
    'lh_parstriangularis', 'lh_pericalcarine', 'lh_postcentral', 'lh_posteriorcingulate',
    'lh_precentral', 'lh_precuneus', 'lh_rostralanteriorcingulate', 'lh_rostralmiddlefrontal',
    'lh_superiorfrontal', 'lh_superiorparietal', 'lh_superiortemporal', 'lh_supramarginal',
    'lh_temporalpole', 'lh_transversetemporal',
    'rh_bankssts', 'rh_caudalanteriorcingulate', 'rh_caudalmiddlefrontal', 'rh_cuneus',
    'rh_entorhinal', 'rh_frontalpole', 'rh_fusiform', 'rh_inferiorparietal',
    'rh_inferiortemporal', 'rh_insula', 'rh_isthmuscingulate', 'rh_lateraloccipital',
    'rh_lateralorbitofrontal', 'rh_lingual', 'rh_medialorbitofrontal', 'rh_middletemporal',
    'rh_parahippocampal', 'rh_paracentral', 'rh_parsopercularis', 'rh_parsorbitalis',
    'rh_parstriangularis', 'rh_pericalcarine', 'rh_postcentral', 'rh_posteriorcingulate',
    'rh_precentral', 'rh_precuneus', 'rh_rostralanteriorcingulate', 'rh_rostralmiddlefrontal',
    'rh_superiorfrontal', 'rh_superiorparietal', 'rh_superiortemporal', 'rh_supramarginal',
    'rh_temporalpole', 'rh_transversetemporal',
]


def load_frequency_vector(file_path, column):
    """Load a frequency CSV and map it to the 68 DK region order."""
    df = pd.read_csv(file_path)
    df.columns = [c.strip() for c in df.columns]
    series = df.set_index('Brain_Region')[column]
    values_68 = np.zeros(68)
    for i, label in enumerate(dk68_labels):
        values_68[i] = series.loc[label] if label in series.index else 0.0
    return values_68


# Shared color scale across cohorts
shared_vmax = 0
for task in tasks:
    if not os.path.exists(task['file']):
        continue
    shared_vmax = max(shared_vmax, load_frequency_vector(task['file'], task['column']).max())
shared_vmax = shared_vmax if shared_vmax > 0 else 1.0

# Render each cohort and compose the 2-row figure
temp_images = []
for task in tasks:
    if not os.path.exists(task['file']):
        continue
    values_68 = load_frequency_vector(task['file'], task['column'])
    temp_path = os.path.join(SAVE_DIR, task['temp_name'])
    values_fsa5 = parcel_to_surface(values_68, 'aparc_fsa5')
    plot_cortical(array_name=values_fsa5, surface_name="fsa5", size=(1200, 300),
                  cmap=custom_cmap_name, color_bar=False,
                  color_range=(0, shared_vmax), screenshot=True,
                  filename=temp_path, background=(1, 1, 1), scale=(3, 3))
    img = plt.imread(temp_path)
    fig_single, ax_s = plt.subplots(figsize=(GLOBAL_CONFIG['FIGURE_WIDTH'], GLOBAL_CONFIG['FIGURE_HEIGHT']))
    ax_s.imshow(img)
    ax_s.axis('off')
    h, w, _ = img.shape
    ax_s.text(w * 0.02, h * 0.1, 'L', fontsize=GLOBAL_CONFIG['FONT_SIZE_LABEL'], fontweight='bold')
    ax_s.text(w * 0.96, h * 0.1, 'R', fontsize=GLOBAL_CONFIG['FONT_SIZE_LABEL'], fontweight='bold')
    plt.savefig(temp_path, bbox_inches='tight', dpi=300)
    plt.close(fig_single)
    temp_images.append(temp_path)

if len(temp_images) == 2:
    fig_final, axes = plt.subplots(2, 1, figsize=(18, 12))
    for i, img_path in enumerate(temp_images):
        axes[i].imshow(plt.imread(img_path))
        axes[i].axis('off')
    cax = fig_final.add_axes([0.92, 0.25, 0.015, 0.5])
    norm = plt.Normalize(vmin=0, vmax=shared_vmax)
    cb = mcolorbar.ColorbarBase(cax, cmap=freq_cmap, norm=norm, orientation='vertical')
    cb.set_ticks([0, shared_vmax])
    cb.set_ticklabels(['0', f'{shared_vmax:.1f}' if shared_vmax < 5 else f'{shared_vmax:.0f}'])
    cb.ax.tick_params(labelsize=GLOBAL_CONFIG['FONT_SIZE_TICK'])
    final_path = os.path.join(SAVE_DIR, 'SubA.png')
    plt.subplots_adjust(left=0.02, right=0.90, hspace=0.05)
    plt.savefig(final_path, bbox_inches='tight', dpi=GLOBAL_CONFIG['DPI'])
    plt.show()
    for f in temp_images:
        os.remove(f)
    print(f"\nCombined figure saved to: {final_path}")

if xvfb_process is not None:
    xvfb_process.terminate()
    xvfb_process.wait()
    print("[Xvfb] closed")
print("Processing Complete.")

## Panel SubB — Frequency maps by subtype

In [ ]:
"""Panel B: cortical map highlighting the 8 core epicenter regions."""
import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from enigmatoolbox.utils.parcellation import parcel_to_surface
from enigmatoolbox.plotting import plot_cortical
from IPython.display import display, Image as IPyImage

# Headless display setup (needed on servers without a display)
os.environ.setdefault('DISPLAY', ':99')
try:
    proc = subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1280x1024x24'],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    proc.poll()
except Exception:
    pass

# --- Global style configuration ---
GLOBAL_CONFIG = {'DPI': 300, 'FIGURE_WIDTH': 12, 'FIGURE_HEIGHT': 10,
                 'FONT_SIZE_MAIN': 20, 'FONT_SIZE_LABEL': 26, 'FONT_SIZE_TICK': 18}
OUTPUT_DIR = 'Fig3'
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, "SubB.png")
plt.rcParams.update({'font.size': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'axes.titlesize': GLOBAL_CONFIG['FONT_SIZE_MAIN'],
                     'font.family': 'serif', 'font.serif': ['Times New Roman'],
                     'savefig.dpi': GLOBAL_CONFIG['DPI']})

# --- Epicenter regions and highlight colormap ---
epicenter_regions = [
    'rh_caudalanteriorcingulate', 'lh_caudalanteriorcingulate',
    'rh_temporalpole', 'lh_temporalpole', 'rh_insula', 'rh_cuneus',
    'rh_frontalpole', 'lh_frontalpole'
]
custom_cmap_name = 'epicenter_blue'
highlight_cmap_obj = LinearSegmentedColormap.from_list(custom_cmap_name, ['#eeeeee', '#084594'])
try:
    plt.colormaps.register(cmap=highlight_cmap_obj, name=custom_cmap_name, force=True)
except Exception:
    pass

# DK68 labels (specific order used by enigmatoolbox fsa5)
dk68_labels = [
    'lh_bankssts', 'lh_caudalanteriorcingulate', 'lh_caudalmiddlefrontal', 'lh_cuneus',
    'lh_entorhinal', 'lh_fusiform', 'lh_inferiorparietal', 'lh_inferiortemporal',
    'lh_isthmuscingulate', 'lh_lateraloccipital', 'lh_lateralorbitofrontal', 'lh_lingual',
    'lh_medialorbitofrontal', 'lh_middletemporal', 'lh_parahippocampal', 'lh_paracentral',
    'lh_parsopercularis', 'lh_parsorbitalis', 'lh_parstriangularis', 'lh_pericalcarine',
    'lh_postcentral', 'lh_posteriorcingulate', 'lh_precentral', 'lh_precuneus',
    'lh_rostralanteriorcingulate', 'lh_rostralmiddlefrontal', 'lh_superiorfrontal',
    'lh_superiorparietal', 'lh_superiortemporal', 'lh_supramarginal', 'lh_frontalpole',
    'lh_temporalpole', 'lh_transversetemporal', 'lh_insula',
    'rh_bankssts', 'rh_caudalanteriorcingulate', 'rh_caudalmiddlefrontal', 'rh_cuneus',
    'rh_entorhinal', 'rh_fusiform', 'rh_inferiorparietal', 'rh_inferiortemporal',
    'rh_isthmuscingulate', 'rh_lateraloccipital', 'rh_lateralorbitofrontal', 'rh_lingual',
    'rh_medialorbitofrontal', 'rh_middletemporal', 'rh_parahippocampal', 'rh_paracentral',
    'rh_parsopercularis', 'rh_parsorbitalis', 'rh_parstriangularis', 'rh_pericalcarine',
    'rh_postcentral', 'rh_posteriorcingulate', 'rh_precentral', 'rh_precuneus',
    'rh_rostralanteriorcingulate', 'rh_rostralmiddlefrontal', 'rh_superiorfrontal',
    'rh_superiorparietal', 'rh_superiortemporal', 'rh_supramarginal', 'rh_frontalpole',
    'rh_temporalpole', 'rh_transversetemporal', 'rh_insula'
]

# --- Build a binary vector: 1 for epicenter regions, 0 otherwise ---
values_68 = np.zeros(68)
for i, label in enumerate(dk68_labels):
    if label in epicenter_regions:
        values_68[i] = 1

# Render the 1x4 base brain image (lateral + medial views)
tmp_path = os.path.join(OUTPUT_DIR, "_tmp_1x4.png")
values_fsa5 = parcel_to_surface(values_68, 'aparc_fsa5')
plot_cortical(array_name=values_fsa5, surface_name="fsa5", size=(1200, 300),
              cmap=custom_cmap_name, color_bar=False, color_range=(0, 1),
              screenshot=True, filename=tmp_path, background=(1, 1, 1), scale=(5, 5))

# Split the 1x4 image into 4 views and rearrange into a 2x2 layout
img = plt.imread(tmp_path)
h_img, w_img, _ = img.shape
view_w = w_img // 4
views = [img[:, i*view_w:(i+1)*view_w] for i in range(4)]


def find_content_range(v):
    gray = v.mean(axis=2)
    rows = np.where(gray.min(axis=1) < 0.98)[0]
    return (rows[0], rows[-1]) if len(rows) else (0, v.shape[0]-1)


ranges = [find_content_range(v) for v in views]
top, bottom = min(r[0] for r in ranges), max(r[1] for r in ranges)
views_cropped = [v[top:bottom+1, :, :3] for v in views]

from PIL import Image, ImageDraw, ImageFont
V_GAP, H_GAP = 10, 8
row1 = np.concatenate([views_cropped[0],
                       np.ones((views_cropped[0].shape[0], H_GAP, 3), dtype=views_cropped[0].dtype),
                       views_cropped[3]], axis=1)
row2 = np.concatenate([views_cropped[1],
                       np.ones((views_cropped[1].shape[0], H_GAP, 3), dtype=views_cropped[1].dtype),
                       views_cropped[2]], axis=1)
combined = np.concatenate([row1, np.ones((V_GAP, row1.shape[1], 3), dtype=row1.dtype), row2], axis=0)

# Normalize to 0-255 uint8
if combined.max() <= 1.0:
    combined = (combined * 255).astype(np.uint8)
else:
    combined = combined.astype(np.uint8)

# Add L/R labels
img_pil = Image.fromarray(combined)
draw = ImageDraw.Draw(img_pil)
try:
    font_large = ImageFont.truetype('/usr/share/fonts/truetype/msttcorefonts/timesbd.ttf', 120)
except Exception:
    font_large = ImageFont.load_default()
h_c, w_c = combined.shape[:2]
half_w, half_h = w_c // 2, h_c // 2
draw.text((int(half_w * 0.04), int(half_h * 0.04)), 'L', fill='black', font=font_large)
draw.text((int(w_c - half_w * 0.04), int(half_h * 0.04)), 'R', fill='black', font=font_large, anchor='rt')
draw.text((int(half_w * 0.04), int(half_h + V_GAP + half_h * 0.04)), 'L', fill='black', font=font_large)
draw.text((int(w_c - half_w * 0.04), int(half_h + V_GAP + half_h * 0.04)), 'R', fill='black', font=font_large, anchor='rt')

img_pil.save(output_path)
print(f"Epicenter distribution map (2x2) saved: {output_path}")

display(IPyImage(output_path))

if os.path.exists(tmp_path):
    os.remove(tmp_path)
print(f"Region-list length: {len(dk68_labels)}")

## Panel SubC — Network connectivity metrics

In [ ]:
"""
Panel C: graph-theoretic topology comparison between epicenter and non-epicenter
regions. Metrics: degree centrality, weighted clustering coefficient, pattern
uniqueness, and local efficiency. Permutation tests + Cohen's d, plotted as
violin + strip plots.
"""
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist, squareform

warnings.filterwarnings("ignore")
plt.rcParams.update({'figure.dpi': 200, 'font.size': 14, 'font.family': 'serif',
                     'font.serif': ['Times New Roman'], 'axes.unicode_minus': False,
                     'axes.edgecolor': 'black', 'axes.linewidth': 1.2,
                     'xtick.color': 'black', 'ytick.color': 'black',
                     'xtick.direction': 'in', 'ytick.direction': 'in'})

TARGET_REGIONS = [
    'rh_caudalanteriorcingulate', 'lh_caudalanteriorcingulate',
    'rh_temporalpole', 'lh_temporalpole', 'rh_insula', 'rh_cuneus',
    'rh_frontalpole', 'lh_frontalpole'
]

# Cohort-specific TDC-average MIND matrices (outputs of the Epicenter notebooks)
MIND_PATHS = {
    'CABIC': 'output/TDC_Average_MIND_CABIC_aparc.csv',
    'ABIDE2': 'output/TDC_Average_MIND_ABIDE2_aparc.csv',
}
N_PERM = 100000
PALETTE = {'Epicenter': '#E74C3C', 'Non-epicenter': '#5DADE2'}


def compute_weighted_clustering(W):
    """Weighted clustering coefficient (Rubinov & Sporns)."""
    W = np.abs(W)
    W = W / W.max()
    n = W.shape[0]
    C = np.zeros(n)
    for i in range(n):
        neighbors = np.where(W[i, :] > 0)[0]
        ki = len(neighbors)
        if ki < 2:
            C[i] = 0
            continue
        ts = 0
        for jj, j in enumerate(neighbors):
            for k in neighbors[jj+1:]:
                if W[j, k] > 0:
                    ts += (W[i, j] * W[i, k] * W[j, k]) ** (1/3)
        C[i] = ts / (ki * (ki - 1))
    return C


def compute_pattern_uniqueness(W):
    """1 minus the mean row-wise connectivity correlation."""
    W = np.abs(W)
    n = W.shape[0]
    cm = np.corrcoef(W)
    return 1 - (cm.sum(axis=1) - 1) / (n - 1)


def compute_local_efficiency(W):
    """Weighted local efficiency (Rubinov & Sporns)."""
    W = np.abs(W)
    n = W.shape[0]
    D = 1.0 / (W + np.eye(n))
    np.fill_diagonal(D, 0)
    El = np.zeros(n)
    for i in range(n):
        nb = np.where(W[i, :] > 0)[0]
        ki = len(nb)
        if ki < 2:
            El[i] = 0
            continue
        sD = D[np.ix_(nb, nb)]
        ns = len(nb)
        eff = 0
        for a in range(ns):
            for b in range(a+1, ns):
                sp = min(sD[a, b], D[nb[a], i] + D[i, nb[b]])
                if sp > 0 and np.isfinite(sp):
                    eff += 1.0 / sp
        El[i] = 2.0 * eff / (ns * (ns - 1))
    return El


all_metric_names = ['Degree Centrality', 'Clustering Coeff', 'Pattern Uniqueness', 'Local Efficiency']
all_metric_funcs = [lambda W: np.abs(W).mean(axis=1), compute_weighted_clustering,
                    compute_pattern_uniqueness, compute_local_efficiency]

results_summary = []
dataset_plot_data = {}

for ds_name in ['CABIC', 'ABIDE2']:
    df_mind = pd.read_csv(MIND_PATHS[ds_name])
    region_names = df_mind.columns.tolist()
    W = df_mind.values
    n_regions = len(region_names)
    target_idx = [region_names.index(r) for r in TARGET_REGIONS if r in region_names]
    metrics_df = pd.DataFrame({'Region': region_names})
    metrics_df['Is_Epi'] = metrics_df['Region'].isin(TARGET_REGIONS)

    for m_name, m_func in zip(all_metric_names, all_metric_funcs):
        values = m_func(W)
        metrics_df[m_name] = values
        epi_vals = values[target_idx]
        non_epi_vals = np.delete(values, target_idx)
        obs_diff = epi_vals.mean() - non_epi_vals.mean()

        # Permutation test: random draw of the same number of regions
        np.random.seed(42)
        perm_means = np.zeros(N_PERM)
        for p in range(N_PERM):
            rand_idx = np.random.choice(n_regions, len(target_idx), replace=False)
            perm_means[p] = values[rand_idx].mean()
        p_less = np.mean(perm_means <= obs_diff)
        p_greater = np.mean(perm_means >= obs_diff)
        p_perm = 2 * min(p_less, p_greater)

        # Cohen's d
        n1, n2 = len(epi_vals), len(non_epi_vals)
        s1, s2 = epi_vals.std(ddof=1), non_epi_vals.std(ddof=1)
        pooled_std = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
        cohens_d = obs_diff / pooled_std

        results_summary.append({'Dataset': ds_name, 'Metric': m_name,
                                'Epi_Mean': epi_vals.mean(), 'Epi_Std': epi_vals.std(ddof=1),
                                'NonEpi_Mean': non_epi_vals.mean(), 'NonEpi_Std': non_epi_vals.std(ddof=1),
                                'Diff': obs_diff, "Cohen's d": cohens_d, 'P_perm': p_perm})
    dataset_plot_data[ds_name] = metrics_df.copy()

# --- Visualization ---
for ds_name in ['CABIC', 'ABIDE2']:
    metrics_df = dataset_plot_data[ds_name]
    melted_df = pd.melt(metrics_df, id_vars=['Region', 'Is_Epi'],
                        value_vars=all_metric_names, var_name='Metric', value_name='Value')
    melted_df['Group'] = melted_df['Is_Epi'].map({True: 'Epicenter', False: 'Non-epicenter'})
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes = axes.flatten()
    for idx, m_name in enumerate(all_metric_names):
        ax = axes[idx]
        sub_df = melted_df[melted_df['Metric'] == m_name]
        sns.violinplot(x='Group', y='Value', data=sub_df, ax=ax, palette=PALETTE,
                       order=['Epicenter', 'Non-epicenter'], inner=None, cut=0,
                       alpha=0.75, linewidth=1.2)
        sns.stripplot(x='Group', y='Value', data=sub_df, ax=ax,
                      order=['Epicenter', 'Non-epicenter'], color='#2C3E50',
                      alpha=0.6, size=6, jitter=0.15)
        ax.set_title('')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.tick_params(axis='both', which='major', labelsize=24, top=True, right=True)
        for sp in ['top', 'right', 'left', 'bottom']:
            ax.spines[sp].set_visible(True)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4, color='#BDC3C7')
        ax.set_axisbelow(True)
    plt.tight_layout(h_pad=5.0)
    save_path = f'Fig3/SubC_{ds_name}.png'
    os.makedirs('Fig3', exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

# --- Summary table (mean +/- std) ---
print("\n" + "="*125)
print(f"{'Dataset':<10} | {'Metric':<22} | {'Epi':<24} | {'NonEpi':<24} | {'Cohen_d':<8} | {'P_perm':<10}")
print("="*125)
for res in results_summary:
    epi_str = f"{res['Epi_Mean']:.4f}±{res['Epi_Std']:.4f}"
    non_str = f"{res['NonEpi_Mean']:.4f}±{res['NonEpi_Std']:.4f}"
    cd = res["Cohen's d"]
    print(f"{res['Dataset']:<10} | {res['Metric']:<22} | {epi_str:<24} | {non_str:<24} | {cd:<8.3f} | {res['P_perm']:<10.4f}")
print("="*125 + "\n")